# Document Processing Pipeline – Execution Notebook

This notebook runs the complete document analysis pipeline developed for the **HMLR Data Science Challenge**.

The pipeline performs the following steps:

1. Convert PDF documents into page images
2. Extract text from images using OCR
3. Classify each page type
4. Extract key entities such as application numbers and applicant names
5. Produce a structured output table

The core processing logic is implemented in the **`src/` modules**, while this notebook simply demonstrates how to run the pipeline end-to-end.

# Environment Setup

Before running the pipeline, we configure the project path so that Python can import the processing modules located in the `src/` directory.

This allows the notebook to reuse the pipeline functions without redefining them here.

In [ ]:
import sys
from pathlib import Path

# Determine project root (parent directory of notebooks/)
PROJECT_ROOT = Path.cwd().parent

# Allow Python to import modules from src/
sys.path.append(str(PROJECT_ROOT))

## Import Processing Pipeline

The main processing function `process_document()` orchestrates the full workflow:

PDF → Images → OCR → Page Classification → Entity Extraction → Structured Output

In [ ]:
from src.pipeline import process_document

## Locate Input Documents

The pipeline expects PDF files to be stored in the `data/raw/` directory.

Any document placed in this folder can be processed without modifying the code.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "raw"

pdf_files = list(DATA_DIR.glob("*.pdf"))

print("PDF files found:", len(pdf_files))
pdf_files

## Run the Pipeline

The selected document is processed through the full pipeline, producing a structured dataset containing:

- page number
- page type
- application numbers
- applicant names

In [ ]:
pdf_path = pdf_files[0]

df_export = process_document(pdf_path)

df_export

In [ ]:
# ---------------------------------------------
# Reshape the output so that each application
# appears on a separate row.
# This format is easier to analyse in Excel.
# ---------------------------------------------

import pandas as pd
rows = []

for _, r in df_export.iterrows():

    apps = r["application_numbers"].split("; ") if r["application_numbers"] else []
    applicants = r["applicants"].split("; ") if r["applicants"] else []

    if not apps:
        rows.append({
            "page": r["page"],
            "page_type": r["page_type"],
            "application_number": "",
            "applicant": ""
        })
        continue

    for i, app in enumerate(apps):

        applicant = applicants[i] if i < len(applicants) else ""

        rows.append({
            "page": r["page"],
            "page_type": r["page_type"],
            "application_number": app,
            "applicant": applicant
        })

df_clean = pd.DataFrame(rows)

df_clean

## Save Results

The final structured output can be saved as a CSV file for downstream analysis or integration into other systems.

In [ ]:
output_file = PROJECT_ROOT / "outputs" / "result.csv"

df_clean.to_csv(output_file, index=False)

print("Results saved to:", output_file)

## Summary

This notebook demonstrates how the modular pipeline implemented in `src/` can be executed on a planning document.

By separating the reusable processing code from the execution notebook, the project remains:

- **modular**
- **easy to maintain**
- **simple to extend for additional documents**

The resulting dataset provides a structured representation of information extracted from scanned planning records.